# 03 — Length of Stay Prediction

Predict whether a patient's LOS will exceed 3 days.

**Task**: Binary (LOS > 3 days: yes/no)  
**Model**: RETAIN  
**Metrics**: PR-AUC, ROC-AUC, F1

In [ ]:
from pyhealth_enterprise.datasets.synthetic import SyntheticEHRDataset
from pyhealth.tasks import length_of_stay_prediction_mimic3_fn
from pyhealth.datasets import split_by_patient, get_dataloader

ds = SyntheticEHRDataset()
ds.load()

task_dataset = ds.dataset.set_task(length_of_stay_prediction_mimic3_fn)
train, val, test = split_by_patient(task_dataset, [0.8, 0.1, 0.1])
train_loader = get_dataloader(train, batch_size=32, shuffle=True)
val_loader   = get_dataloader(val,   batch_size=32, shuffle=False)
test_loader  = get_dataloader(test,  batch_size=32, shuffle=False)

In [ ]:
from pyhealth.models import RETAIN
from pyhealth.trainer import Trainer

model = RETAIN(
    dataset=task_dataset,
    feature_keys=['conditions', 'drugs'],
    label_key='los',
    mode='binary',
)
trainer = Trainer(model=model, metrics=['pr_auc', 'roc_auc', 'f1'])
trainer.train(train_dataloader=train_loader, val_dataloader=val_loader,
              epochs=50, monitor='pr_auc')
print(trainer.evaluate(test_loader))